# Решения: preprocessing pipeline

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден — положите slim CSV рядом с ноутбуком')


ORDERS_PATH = _find('orders_slim.csv')
CUSTOMERS_PATH = _find('customers_slim.csv')
PAYMENTS_PATH = _find('payments_slim.csv')

orders = pd.read_csv(ORDERS_PATH, parse_dates=['order_purchase_timestamp'])
if 'order_delivered_customer_date' in orders.columns:
    orders['order_delivered_customer_date'] = pd.to_datetime(
        orders['order_delivered_customer_date'], errors='coerce'
    )
customers = pd.read_csv(CUSTOMERS_PATH)
payments = pd.read_csv(PAYMENTS_PATH)


## Урок. 1-5

In [ ]:
def preprocess_customers(orders_df, customers_df, payments_df):
    log = []
    orders_local = orders_df.copy()
    customers_local = customers_df.copy()
    payments_local = payments_df.copy()
    log.append('copied input frames')
    if (payments_local['payment_value'] < 0).any():
        raise ValueError('negative payment_value in payments')
    if orders_local['order_purchase_timestamp'].isna().any():
        raise ValueError('missing order_purchase_timestamp in orders')
    log.append('validated input contracts')
    merged = orders_local.merge(payments_local, on='order_id', how='left')
    merged = merged.merge(customers_local[['customer_id', 'customer_state']], on='customer_id', how='left')
    log.append('merged orders, payments, customers')
    merged['days_to_deliver'] = (
        merged['order_delivered_customer_date'] - merged['order_purchase_timestamp']
    ).dt.days
    ref_date = merged['order_purchase_timestamp'].max()
    base = (
        merged.groupby('customer_id')
        .agg(
            last_purchase=('order_purchase_timestamp', 'max'),
            Frequency=('order_id', 'nunique'),
            Monetary=('payment_value', 'sum'),
            avg_days_to_deliver=('days_to_deliver', 'mean'),
        )
        .reset_index()
    )
    base['Recency'] = (ref_date - base['last_purchase']).dt.days
    share_card = merged.groupby('customer_id')['payment_type'].apply(
        lambda s: float((s == 'credit_card').mean())
    )
    features = base.merge(share_card.rename('share_card'), on='customer_id', how='left')
    features = features[['customer_id', 'Recency', 'Frequency', 'Monetary', 'share_card', 'avg_days_to_deliver']]
    log.append('built customer features RFM + extras')
    return features, log


features, log = preprocess_customers(orders, customers, payments)
required_cols = ['customer_id', 'Recency', 'Frequency', 'Monetary', 'share_card', 'avg_days_to_deliver']
acceptance = pd.Series(
    [
        all(c in features.columns for c in required_cols),
        {'share_card', 'avg_days_to_deliver'} <= set(features.columns),
        any('validated' in step for step in log),
        len(log) >= 4,
        False,
    ],
    index=['has_rfm', 'has_extra_features', 'validated_input', 'has_log', 'saved_preview'],
)
preview_path = Path('features_preview.csv')
features.to_csv(preview_path, index=False)
acceptance.loc['saved_preview'] = preview_path.exists()
REPORT = (
    f'Собран preprocessing для {len(features)} клиентов на slim-данных заказов. '
    'Контракт входа проверяет отрицательные оплаты и пропуски даты покупки. '
    'На выходе таблица клиентских признаков: Recency, Frequency, Monetary, доля card и средняя задержка доставки. '
    'Лог фиксирует ключевые шаги и позволяет воспроизвести результат. '
    'Preview сохранён в features_preview.csv; модель в этом модуле не обучается.'
)
READY = bool(acceptance.all())
print(features.head())
print(log)
print(acceptance)
print('READY:', READY)

## ДЗ. 1-3

In [ ]:
def preprocess_customers(orders_df, customers_df, payments_df):
    log = []
    orders_local = orders_df.copy()
    payments_local = payments_df.copy()
    if (payments_local['payment_value'] < 0).any():
        raise ValueError('negative payment_value')
    log.append('validated payments')
    merged = orders_local.merge(payments_local, on='order_id', how='left')
    log.append('merged orders-payments')
    ref_date = merged['order_purchase_timestamp'].max()
    features = (
        merged.groupby('customer_id')
        .agg(last_purchase=('order_purchase_timestamp', 'max'), Frequency=('order_id', 'nunique'), Monetary=('payment_value', 'sum'))
        .reset_index()
    )
    features['Recency'] = (ref_date - features['last_purchase']).dt.days
    features = features[['customer_id', 'Recency', 'Frequency', 'Monetary']]
    log.append('built base RFM')
    return features, log


features, log = preprocess_customers(orders, customers, payments)
features2, _ = preprocess_customers(orders, customers, payments)
same_shape = features.shape == features2.shape
NEXT_NOTE = (
    'В следующий модуль стоит вынести масштабирование и выбор признаков под конкретную модель, '
    'а также автоматическую проверку качества на train/validation. '
    'Текущий шаг уже даёт стабильную инженерную заготовку признаков.'
)
print(features.shape, log)
print('same shape:', same_shape)
print(NEXT_NOTE)